In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## Importing KYC data

In [2]:
# 1. Define your base directory using Pathlib (makes it easy to update later)
BASE_DIR = Path("/Users/wmuheki/Documents/Projects/Analytics/kyc/clean_dumps")

# 2. Load only two DataFrames to save massive amounts of RAM  ---- Change Dataframe Names ‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️
df = pd.read_parquet(BASE_DIR / "df_03_26.parquet")
df_NID = pd.read_parquet(BASE_DIR / "df_NID_03_26.parquet")

In [3]:
# Keep only rows where the MSISDN is exactly 10 characters long
df_NID = df_NID[df_NID['msisdn'].astype(str).str.len() == 10]
df = df[df['msisdn'].astype(str).str.len() == 10]

In [4]:
# Ensure the column is treated as text, then replace the leading '0' with '256'
df_NID['msisdn'] = df_NID['msisdn'].astype(str).str.replace(r'^0', '256', regex=True)
df['msisdn'] = df['msisdn'].astype(str).str.replace(r'^0', '256', regex=True)

In [5]:
# 3. Create lightweight 'views' dynamically based on cleaned 'id_type' column
df_PASS = df[df["id_type"] == "PASSPORT"].reset_index(drop=True)
df_REF = df[df["id_type"] == "REFUGEE_ID"].reset_index(drop=True)
df_COM = df[df["id_type"].isin(["EMPLOYEE_ID", "COMPANY_ID"])].reset_index(drop=True)

In [6]:
df_NID.head()

,msisdn,first_name,surname,id_type,id_number,prefix,mno,gender,birth_year,age,district
0,256071607211,NUBUWATI,LWANGA,NATIONAL_ID,CF92098105FTWE,071,UTCL,Female,1992,34,BUKOMANSIMBI
1,256071607212,NUBUWATI,LWANGA,NATIONAL_ID,CF92098105FTWE,071,UTCL,Female,1992,34,BUKOMANSIMBI
2,256071607213,NUBUWATI,LWANGA,NATIONAL_ID,CF92098105FTWE,071,UTCL,Female,1992,34,BUKOMANSIMBI
3,256071607214,NUBUWATI,LWANGA,NATIONAL_ID,CF92098105FTWE,071,UTCL,Female,1992,34,BUKOMANSIMBI
4,256411671910,DAVID,WASSWA,NATIONAL_ID,CM7305210F16FL,04,UTCL,Male,1973,53,WAKISO


In [7]:
df_REF.head()

,msisdn,first_name,surname,id_type,id_number,prefix,mno
0,256414671450,GHABOUSH,TARIG,REFUGEE_ID,H9B-104641452,04,UTCL
1,256481660372,CHRIS,NGOSAMA,REFUGEE_ID,33500009888,04,UTCL
2,256711003195,GHEBREMEDHIN,TESFAY,REFUGEE_ID,441-00001210,071,UTCL
3,256711157611,BIGIRIMANA,ANICENT,REFUGEE_ID,662-00029396,071,UTCL
4,256711283653,SAID,AHMED,REFUGEE_ID,662-00007786,071,UTCL


In [8]:
df_COM.head()

,msisdn,first_name,surname,id_type,id_number,prefix,mno
0,256727599310,Agents_kampala447,,COMPANY_ID,72064,0727,LYCA
1,256727524784,agents_fortportal675,,COMPANY_ID,72064,0727,LYCA
2,256727289545,agents12,,COMPANY_ID,72064,0727,LYCA
3,256727244219,Agents_kampala447,,COMPANY_ID,72064,0727,LYCA
4,256727356125,wheels_investments,,COMPANY_ID,80020002052652,0727,LYCA


In [9]:
df_PASS.head()

,msisdn,first_name,surname,id_type,id_number,prefix,mno
0,256414234960,ABDUL NAZER,SHAHUL,PASSPORT,L3337196,04,UTCL
1,256414237006,ABOUBACAR,KOUROUMA,PASSPORT,000252637,04,UTCL
2,256414252482,KALPESH,BHATT,PASSPORT,Z5639353,04,UTCL
3,256414252918,SALIM,ALLANI,PASSPORT,Z5126973,04,UTCL
4,256414252919,SALIM,ALLANI,PASSPORT,Z5126973,04,UTCL


## Importing GSMA data

In [10]:
import clickhouse_connect

# Connect to ClickHouse with resource limits
client = clickhouse_connect.get_client(
    host='192.168.1.95',
    port=8123,
    username='default',
    password='',
    database='ceir',
    settings={
        'max_memory_usage': 4000000000,  # 4GB max
        'max_threads': 2,
        'priority': 5
    }
)

# Step 1: Fetch data from the gsma_devices table
gsma_query = """
SELECT
    tac,
    oem,
    brand,
    model,
    marketing_name,
    device_type,
    os_family,
    os_version,
    sim_slots,
    has_2g,
    has_3g,
    has_4g,
    has_5g,
    year_released
FROM gsma_devices
"""
gsma_df = client.query_df(gsma_query)


In [11]:
len(gsma_df)

290402

In [12]:
gsma_df.head()

,tac,oem,brand,model,marketing_name,device_type,os_family,os_version,sim_slots,has_2g,has_3g,has_4g,has_5g,year_released
0,35697403,Not Known,Not Known,ROWEL K658,,Handheld,,,0,0,0,0,0,0
1,35697404,Not Known,G crown,"G265, G765, G865, G965",,Handheld,,,0,0,0,0,0,0
2,35697405,Not Known,QMobile,Q4,Q4 TV,Handheld,Other,,0,1,0,0,0,2013
3,35697406,Not Known,Apple,iPad mini (A1600),iPad mini 3,Tablet,iOS,8_1,1,1,1,1,0,2014
4,35697407,Not Known,TC,TC F6,,Mobile Phone/Feature phone,,,0,0,0,0,0,0


## Importing IMEI, IMSI and TIMESTAMP Data Dumps from CEIR - ClickHouse Bronze

In [13]:
# Load imei, imsi, timestamp data from CSV files (these are dumps generated from clickhouse Bronze)

domestic_bronze = pd.read_csv(
    "/Users/wmuheki/Documents/Projects/Analytics/ceir/dumps/domestic_dedup_earliest.csv",
    dtype={"imei": str, "imsi": str},
    parse_dates=["timestamp"]
)

roamers_bronze = pd.read_csv(
    "/Users/wmuheki/Documents/Projects/Analytics/ceir/dumps/roamers_dedup_earliest.csv",
    dtype={"imei": str, "imsi": str},
    parse_dates=["timestamp"]
)

In [14]:
roamers_bronze = roamers_bronze[["timestamp", "imei", "imsi"]]
roamers_bronze = roamers_bronze.rename(columns={"timestamp": "first_seen"})

roamers_bronze.head()

,first_seen,imei,imsi
0,2025-08-25 19:04:23,35645083285531,630020332356816
1,2025-08-11 21:28:25,35011629497137,635130144500118
2,2026-01-01 10:38:39,35380989554540,639035035961782
3,2026-02-13 20:41:27,35216677019973,635130149032390
4,2025-10-08 23:12:38,86746105046960,639035039638374


In [15]:
domestic_bronze = domestic_bronze[["timestamp", "imei", "imsi"]]
domestic_bronze = domestic_bronze.rename(columns={"timestamp": "first_seen"})

domestic_bronze.head()

,first_seen,imei,imsi
0,2025-11-03 15:02:24,86987003394320,641010261297359
1,2025-09-04 14:40:14,35561976146038,641101929503549
2,2025-09-17 16:03:18,35402511219948,641010408639500
3,2025-11-03 19:04:38,35081458532431,641010417473235
4,2025-12-25 23:27:28,35621860194963,641010273983767


In [16]:
len(roamers_bronze)

9709585

In [17]:
len(domestic_bronze)

193544286

## Importing Data from Gold Layer - We Use it to extract Corresponding MSISDNs

In [18]:
import clickhouse_connect

# Connect to ClickHouse with resource limits
client = clickhouse_connect.get_client(
    host='192.168.1.95',
    port=8123,
    username='default',
    password='',
    database='ceir_gold',
    settings={
        'max_memory_usage': 4000000000,  # 4GB max
        'max_threads': 2,
        'priority': 5
    }
)

# Step 1: Fetch data from the domestic_subscribers table
domestic_query = """
SELECT
    last_seen,
    msisdn,
    imsi
FROM domestic_subscribers
"""
domestic_gold = client.query_df(domestic_query)


# Step 2: Fetch data from the roamers table
roamers_query = """
SELECT
    last_seen,
    msisdn,
    imsi
FROM roamers
"""
roamers_gold = client.query_df(roamers_query)

In [19]:
domestic_gold = domestic_gold[["last_seen", "msisdn", "imsi"]]

domestic_gold.head()

,last_seen,msisdn,imsi
0,2026-01-08 22:05:28,11831532509,641010414762525
1,2026-01-02 15:38:01,11831613467,641010408696074
2,2026-01-11 23:27:04,11831613475,641010417545598
3,2025-12-28 10:29:23,11831613476,641010268444940
4,2025-10-19 19:11:39,11831613481,641010402667384


In [20]:
roamers_gold = roamers_gold[["last_seen", "msisdn", "imsi"]]

roamers_gold.head()

,last_seen,msisdn,imsi
0,2025-10-26 21:13:04,,202010903843522
1,2025-12-01 04:42:06,306974636058,202010904271124
2,2026-02-09 11:51:59,,202010904702122
3,2026-02-18 14:15:41,,202010904820283
4,2026-02-09 04:27:18,,202010905099535


In [21]:
len(domestic_gold)

58778465

In [22]:
len(roamers_gold)

7956406

## Merge Bronze and Gold Datasets

In [23]:
# Merge df_gold into df_bronze based on the 'imsi' column
domestic_merged = domestic_bronze.merge(domestic_gold, on='imsi', how='left')
roamers_merged = roamers_bronze.merge(roamers_gold, on='imsi', how='left')

In [24]:
# Convert IMEI to string, extract the first 8 characters, and create the 'tac' column
domestic_merged['tac'] = domestic_merged['imei'].astype(str).str[:8]
roamers_merged['tac'] = roamers_merged['imei'].astype(str).str[:8]

In [25]:
domestic_merged.head()

,first_seen,imei,imsi,last_seen,msisdn,tac
0,2025-11-03 15:02:24,86987003394320,641010261297359,2026-02-10 22:21:47,256709883187,86987003
1,2025-09-04 14:40:14,35561976146038,641101929503549,2026-02-18 00:08:47,256761844456,35561976
2,2025-09-17 16:03:18,35402511219948,641010408639500,2026-02-25 18:01:55,256743403563,35402511
3,2025-09-17 16:03:18,35402511219948,641010408639500,2026-03-08 15:02:48,256743403563,35402511
4,2025-11-03 19:04:38,35081458532431,641010417473235,2025-12-09 10:50:34,256753395881,35081458


In [26]:
roamers_merged.head()

,first_seen,imei,imsi,last_seen,msisdn,tac
0,2025-08-25 19:04:23,35645083285531,630020332356816,2026-01-16 19:43:05,243981492588,35645083
1,2025-08-11 21:28:25,35011629497137,635130144500118,2026-03-12 19:23:51,250721707251,35011629
2,2025-08-11 21:28:25,35011629497137,635130144500118,2026-04-10 18:16:55,,35011629
3,2025-08-11 21:28:25,35011629497137,635130144500118,2026-02-24 20:42:24,250721707251,35011629
4,2026-01-01 10:38:39,35380989554540,639035035961782,2026-03-04 09:39:02,254787969731,35380989


In [27]:
len(domestic_merged)

244992196

In [28]:
len(roamers_merged)

16965677

## Merge with GSMA Data

In [29]:
ceir_domestic = domestic_merged.merge(gsma_df, on='tac', how='left')

In [30]:
#Change Data type to remove decimal points and convert to integers
ceir_domestic['sim_slots'] = ceir_domestic['sim_slots'].astype('Int64')
ceir_domestic['has_2g'] = ceir_domestic['has_2g'].astype('Int64')
ceir_domestic['has_3g'] = ceir_domestic['has_3g'].astype('Int64')
ceir_domestic['has_4g'] = ceir_domestic['has_4g'].astype('Int64')
ceir_domestic['has_5g'] = ceir_domestic['has_5g'].astype('Int64')
ceir_domestic['year_released'] = ceir_domestic['year_released'].astype('Int64')

In [31]:
ceir_domestic.head()

,first_seen,imei,imsi,last_seen,msisdn,tac,oem,brand,model,marketing_name,device_type,os_family,os_version,sim_slots,has_2g,has_3g,has_4g,has_5g,year_released
0,2025-11-03 15:02:24,86987003394320,641010261297359,2026-02-10 22:21:47,256709883187,86987003,Vivo Mobile Communication Co Ltd,vivo,vivo Y85,Y85,Smartphone,Android,8.1,0,1,1,1,0,2018
1,2025-11-03 15:02:24,86987003394320,641010261297359,2026-02-10 22:21:47,256709883187,86987003,Vivo Mobile Communication Co Ltd,vivo,vivo Y85,Y85,Smartphone,Android,8.1,0,1,1,1,0,2018
2,2025-09-04 14:40:14,35561976146038,641101929503549,2026-02-18 00:08:47,256761844456,35561976,INFINIX TECHNOLOGY LIMITED,Infinix,X6525D,Smart 8/Smart 10 HD,Smartphone,Android,14,2,1,1,1,0,2025
3,2025-09-17 16:03:18,35402511219948,641010408639500,2026-02-25 18:01:55,256743403563,35402511,Tecno Telecom (HK) Limited,TECNO,B1f,Pop 2F,Smartphone,Android,8.1,2,1,1,0,0,2019
4,2025-09-17 16:03:18,35402511219948,641010408639500,2026-03-08 15:02:48,256743403563,35402511,Tecno Telecom (HK) Limited,TECNO,B1f,Pop 2F,Smartphone,Android,8.1,2,1,1,0,0,2019


In [32]:
len(ceir_domestic)

248950597

In [33]:
ceir_roamers = roamers_merged.merge(gsma_df, on='tac', how='left')

In [34]:
#Change Data type to remove decimal points and convert to integers
ceir_roamers['sim_slots'] = ceir_roamers['sim_slots'].astype('Int64')
ceir_roamers['has_2g'] = ceir_roamers['has_2g'].astype('Int64')
ceir_roamers['has_3g'] = ceir_roamers['has_3g'].astype('Int64')
ceir_roamers['has_4g'] = ceir_roamers['has_4g'].astype('Int64')
ceir_roamers['has_5g'] = ceir_roamers['has_5g'].astype('Int64')
ceir_roamers['year_released'] = ceir_roamers['year_released'].astype('Int64')

In [35]:
ceir_roamers.head()

,first_seen,imei,imsi,last_seen,msisdn,tac,oem,brand,model,marketing_name,device_type,os_family,os_version,sim_slots,has_2g,has_3g,has_4g,has_5g,year_released
0,2025-08-25 19:04:23,35645083285531,630020332356816,2026-01-16 19:43:05,243981492588,35645083,Tecno Telecom (HK) Limited,TECNO,BE7,Pop 6 LTE,Smartphone,Android,11,2,1,1,1,0,2022
1,2025-08-11 21:28:25,35011629497137,635130144500118,2026-03-12 19:23:51,250721707251,35011629,Itel Technology Limited,itel,it2160,it2160,Mobile Phone/Feature phone,,,2,1,0,0,0,2018
2,2025-08-11 21:28:25,35011629497137,635130144500118,2026-04-10 18:16:55,,35011629,Itel Technology Limited,itel,it2160,it2160,Mobile Phone/Feature phone,,,2,1,0,0,0,2018
3,2025-08-11 21:28:25,35011629497137,635130144500118,2026-02-24 20:42:24,250721707251,35011629,Itel Technology Limited,itel,it2160,it2160,Mobile Phone/Feature phone,,,2,1,0,0,0,2018
4,2026-01-01 10:38:39,35380989554540,639035035961782,2026-03-04 09:39:02,254787969731,35380989,Samsung Korea,Samsung,SM-A047F/DS,Galaxy A04s,Smartphone,Android,12,2,1,1,1,0,2022


In [36]:
len(ceir_roamers)

17407790

## Enrich Roamers Dataset with Country of SIM Origin

In [37]:
mcc_lu = pd.read_csv("/Users/wmuheki/Documents/Projects/Analytics/CEIR/clean_dumps/MCC_Each_country.csv", dtype=str)
mcc_lu.head()

,MCC,Country
0,289,Abkhazia
1,412,Afghanistan
2,276,Albania
3,603,Algeria
4,544,American Samoa


In [38]:
# 1) Normalize lookup columns
mcc_lu = mcc_lu.rename(columns={"MCC": "mcc", "Country": "country"})
mcc_lu["mcc"] = mcc_lu["mcc"].astype("string").str.strip()
mcc_lu["country"] = mcc_lu["country"].astype("string").str.strip()
mcc_lu = mcc_lu.drop_duplicates(subset=["mcc"])


# 2) Extract MCC from IMSI (first 3 digits) - for  roaming data
ceir_roamers["imsi"] = ceir_roamers["imsi"].astype("string")

ceir_roamers["mcc"] = (
    ceir_roamers["imsi"]
    .str.replace(r"\D+", "", regex=True)  # keep digits only
    .str.slice(0, 3)
)

# 3) Merge Country Code Data Frame with the Roaming Data sets
ceir_roamers = ceir_roamers.merge(
    mcc_lu[["mcc", "country"]],
    how="left",
    on="mcc"
)

# 4) Optional: fill unknowns
ceir_roamers["country"] = ceir_roamers["country"].fillna("UNKNOWN")

In [39]:
ceir_roamers.head()

,first_seen,imei,imsi,last_seen,msisdn,tac,oem,brand,model,marketing_name,...,os_family,os_version,sim_slots,has_2g,has_3g,has_4g,has_5g,year_released,mcc,country
0,2025-08-25 19:04:23,35645083285531,630020332356816,2026-01-16 19:43:05,243981492588,35645083,Tecno Telecom (HK) Limited,TECNO,BE7,Pop 6 LTE,...,Android,11,2,1,1,1,0,2022,630,Democratic Republic of Congo
1,2025-08-11 21:28:25,35011629497137,635130144500118,2026-03-12 19:23:51,250721707251,35011629,Itel Technology Limited,itel,it2160,it2160,...,,,2,1,0,0,0,2018,635,Rwanda
2,2025-08-11 21:28:25,35011629497137,635130144500118,2026-04-10 18:16:55,,35011629,Itel Technology Limited,itel,it2160,it2160,...,,,2,1,0,0,0,2018,635,Rwanda
3,2025-08-11 21:28:25,35011629497137,635130144500118,2026-02-24 20:42:24,250721707251,35011629,Itel Technology Limited,itel,it2160,it2160,...,,,2,1,0,0,0,2018,635,Rwanda
4,2026-01-01 10:38:39,35380989554540,639035035961782,2026-03-04 09:39:02,254787969731,35380989,Samsung Korea,Samsung,SM-A047F/DS,Galaxy A04s,...,Android,12,2,1,1,1,0,2022,639,Kenya


## Merge Domestic Data with KYC data

In [40]:
import pandas as pd

# STEP 1: Stack the four registration datasets vertically
# DataFrames like df_REF that lack 'gender' or 'district' will automatically get NaN in those columns
dataframes_to_combine = [df_NID, df_REF, df_COM, df_PASS]
df_master_reg = pd.concat(dataframes_to_combine, ignore_index=True)

# STEP 2: Standardize the joining key to string to prevent silent merge failures
ceir_domestic['msisdn'] = ceir_domestic['msisdn'].astype(str)
df_master_reg['msisdn'] = df_master_reg['msisdn'].astype(str)

# STEP 3: Perform a single left merge onto your CEIR dataset
df_final = ceir_domestic.merge(df_master_reg, on='msisdn', how='left')

# Preview the results
df_final.head()

,first_seen,imei,imsi,last_seen,msisdn,tac,oem,brand,model,marketing_name,...,first_name,surname,id_type,id_number,prefix,mno,gender,birth_year,age,district
0,2025-11-03 15:02:24,86987003394320,641010261297359,2026-02-10 22:21:47,256709883187,86987003,Vivo Mobile Communication Co Ltd,vivo,vivo Y85,Y85,...,MARIA ASUMPTA,NABBANJA,NATIONAL_ID,CF48068106UXWE,070,AIRTEL,Female,1948,78,MITYANA
1,2025-11-03 15:02:24,86987003394320,641010261297359,2026-02-10 22:21:47,256709883187,86987003,Vivo Mobile Communication Co Ltd,vivo,vivo Y85,Y85,...,MARIA ASUMPTA,NABBANJA,NATIONAL_ID,CF48068106UXWE,070,AIRTEL,Female,1948,78,MITYANA
2,2025-09-04 14:40:14,35561976146038,641101929503549,2026-02-18 00:08:47,256761844456,35561976,INFINIX TECHNOLOGY LIMITED,Infinix,X6525D,Smart 8/Smart 10 HD,...,SEDRACK,OMUJAL,NATIONAL_ID,CM9902110627JD,076,MTN,Male,1999,27,KUMI
3,2025-09-17 16:03:18,35402511219948,641010408639500,2026-02-25 18:01:55,256743403563,35402511,Tecno Telecom (HK) Limited,TECNO,B1f,Pop 2F,...,JENIPHER,KHAINZA,NATIONAL_ID,CF68067100KLQD,074,AIRTEL,Female,1968,58,MANAFWA
4,2025-09-17 16:03:18,35402511219948,641010408639500,2026-03-08 15:02:48,256743403563,35402511,Tecno Telecom (HK) Limited,TECNO,B1f,Pop 2F,...,JENIPHER,KHAINZA,NATIONAL_ID,CF68067100KLQD,074,AIRTEL,Female,1968,58,MANAFWA


In [41]:
len(df_final)

248950597

### Enrich with MNO for Domestic Dataset

In [42]:
# We first drop the existing 'mno' column to avoid confusion
df_final.drop(columns=['mno'], inplace=True)

# Ensure IMSI is string (important)
df_final["imsi"] = df_final["imsi"].astype(str)


# Derive MNO from IMSI prefix
df_final["mno"] = (
    df_final["imsi"]
    .str[:5]
    .map({
        "64110": "MTN",
        "64101": "AIRTEL",
        "64122": "AIRTEL",
        "64120": "HAMILTON",
        "64108": "TALKIO",
    })
    .fillna("UNKNOWN")
)

In [43]:
df_final.head()

,first_seen,imei,imsi,last_seen,msisdn,tac,oem,brand,model,marketing_name,...,first_name,surname,id_type,id_number,prefix,gender,birth_year,age,district,mno
0,2025-11-03 15:02:24,86987003394320,641010261297359,2026-02-10 22:21:47,256709883187,86987003,Vivo Mobile Communication Co Ltd,vivo,vivo Y85,Y85,...,MARIA ASUMPTA,NABBANJA,NATIONAL_ID,CF48068106UXWE,070,Female,1948,78,MITYANA,AIRTEL
1,2025-11-03 15:02:24,86987003394320,641010261297359,2026-02-10 22:21:47,256709883187,86987003,Vivo Mobile Communication Co Ltd,vivo,vivo Y85,Y85,...,MARIA ASUMPTA,NABBANJA,NATIONAL_ID,CF48068106UXWE,070,Female,1948,78,MITYANA,AIRTEL
2,2025-09-04 14:40:14,35561976146038,641101929503549,2026-02-18 00:08:47,256761844456,35561976,INFINIX TECHNOLOGY LIMITED,Infinix,X6525D,Smart 8/Smart 10 HD,...,SEDRACK,OMUJAL,NATIONAL_ID,CM9902110627JD,076,Male,1999,27,KUMI,MTN
3,2025-09-17 16:03:18,35402511219948,641010408639500,2026-02-25 18:01:55,256743403563,35402511,Tecno Telecom (HK) Limited,TECNO,B1f,Pop 2F,...,JENIPHER,KHAINZA,NATIONAL_ID,CF68067100KLQD,074,Female,1968,58,MANAFWA,AIRTEL
4,2025-09-17 16:03:18,35402511219948,641010408639500,2026-03-08 15:02:48,256743403563,35402511,Tecno Telecom (HK) Limited,TECNO,B1f,Pop 2F,...,JENIPHER,KHAINZA,NATIONAL_ID,CF68067100KLQD,074,Female,1968,58,MANAFWA,AIRTEL


## Split into Monthly Datasets

In [44]:
# 1. Ensure 'first_seen' is a datetime data type
# (It might already be, but this guarantees it so we can use .dt accessors)
ceir_roamers['first_seen'] = pd.to_datetime(ceir_roamers['first_seen'])

# 2. Initialize an empty dictionary to store your monthly datasets
roaming_datasets = {}

# 3. Group the dataframe by the Year and Month
# .dt.to_period('M') converts the timestamp to a monthly period (e.g., 2025-11)
for month_period, group_df in ceir_roamers.groupby(ceir_roamers['first_seen'].dt.to_period('M')):
    
    # Convert the period to a string to use as the dictionary key
    month_str = str(month_period) 
    
    # Store the group in the dictionary
    # .copy() ensures it's an independent dataframe, not a slice/view of the original
    roaming_datasets[month_str] = group_df.copy() 

print(f"Created Roamer datasets for the following months: {list(roaming_datasets.keys())}")

Created Roamer datasets for the following months: ['2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04']


In [45]:
# Extract the dataset for Each MOnth
roaming_2025_08 = roaming_datasets['2025-08']
roaming_2025_09 = roaming_datasets['2025-09']
roaming_2025_10 = roaming_datasets['2025-10']
roaming_2025_11 = roaming_datasets['2025-11']
roaming_2025_12 = roaming_datasets['2025-12']
roaming_2026_01 = roaming_datasets['2026-01']
roaming_2026_02 = roaming_datasets['2026-02']
roaming_2026_03 = roaming_datasets['2026-03']
roaming_2026_04 = roaming_datasets['2026-04']


In [46]:
import pandas as pd

# 1. Ensure 'first_seen' is a datetime data type
# (It might already be, but this guarantees it so we can use .dt accessors)
df_final['first_seen'] = pd.to_datetime(df_final['first_seen'])

# 2. Initialize an empty dictionary to store your monthly datasets
domestic_datasets = {}

# 3. Group the dataframe by the Year and Month
# .dt.to_period('M') converts the timestamp to a monthly period (e.g., 2025-11)
for month_period, group_df in df_final.groupby(df_final['first_seen'].dt.to_period('M')):
    
    # Convert the period to a string to use as the dictionary key
    month_str = str(month_period) 
    
    # Store the group in the dictionary
    # .copy() ensures it's an independent dataframe, not a slice/view of the original
    domestic_datasets[month_str] = group_df.copy() 

print(f"Created Domestic datasets for the following months: {list(domestic_datasets.keys())}")

Created Domestic datasets for the following months: ['2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03']


In [47]:
# Extract the dataset for Each MOnth
domestic_2025_09 = domestic_datasets['2025-09']
domestic_2025_10 = domestic_datasets['2025-10']
domestic_2025_11 = domestic_datasets['2025-11']
domestic_2025_12 = domestic_datasets['2025-12']
domestic_2026_01 = domestic_datasets['2026-01']
domestic_2026_02 = domestic_datasets['2026-02']
domestic_2026_03 = domestic_datasets['2026-03']

## Export Datasets

In [48]:
roaming_2025_08.to_parquet("/Users/wmuheki/Documents/Projects/Analytics/ceir/dumps/roaming_2025_08.parquet") # Change file name as needed
roaming_2025_09.to_parquet("/Users/wmuheki/Documents/Projects/Analytics/ceir/dumps/roaming_2025_09.parquet")
roaming_2025_10.to_parquet("/Users/wmuheki/Documents/Projects/Analytics/ceir/dumps/roaming_2025_10.parquet")
roaming_2025_11.to_parquet("/Users/wmuheki/Documents/Projects/Analytics/ceir/dumps/roaming_2025_11.parquet")
roaming_2025_12.to_parquet("/Users/wmuheki/Documents/Projects/Analytics/ceir/dumps/roaming_2025_12.parquet")
roaming_2026_01.to_parquet("/Users/wmuheki/Documents/Projects/Analytics/ceir/dumps/roaming_2026_01.parquet")
roaming_2026_02.to_parquet("/Users/wmuheki/Documents/Projects/Analytics/ceir/dumps/roaming_2026_02.parquet")
roaming_2026_03.to_parquet("/Users/wmuheki/Documents/Projects/Analytics/ceir/dumps/roaming_2026_03.parquet")
roaming_2026_04.to_parquet("/Users/wmuheki/Documents/Projects/Analytics/ceir/dumps/roaming_2026_04.parquet")

In [49]:
domestic_2025_09.to_parquet("/Users/wmuheki/Documents/Projects/Analytics/ceir/dumps/domestic_2025_09.parquet") # Change file name as needed
domestic_2025_10.to_parquet("/Users/wmuheki/Documents/Projects/Analytics/ceir/dumps/domestic_2025_10.parquet")
domestic_2025_11.to_parquet("/Users/wmuheki/Documents/Projects/Analytics/ceir/dumps/domestic_2025_11.parquet")
domestic_2025_12.to_parquet("/Users/wmuheki/Documents/Projects/Analytics/ceir/dumps/domestic_2025_12.parquet")
domestic_2026_01.to_parquet("/Users/wmuheki/Documents/Projects/Analytics/ceir/dumps/domestic_2026_01.parquet")
domestic_2026_02.to_parquet("/Users/wmuheki/Documents/Projects/Analytics/ceir/dumps/domestic_2026_02.parquet")
domestic_2026_03.to_parquet("/Users/wmuheki/Documents/Projects/Analytics/ceir/dumps/domestic_2026_03.parquet")